In [16]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv


# Customer Churn Analysis - Week 1 EDA
**Author**: [Rabia Hafeez]
**Date**: [04/03/2026]
**Course**: Introduction to Applied AI
## Project Overview
This notebook performs exploratory data analysis on customer churn data to identify
patterns and inform our predictive modeling approach.
## Table of Contents
1. Dataset Overview
2. Numerical Features Analysis
3. Categorical Features Analysis
4. Feature Correlations
5. Key Insights and Findings

In [17]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("blastchar/telco-customer-churn")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/blastchar/telco-customer-churn


In [ ]:
pip install pandas numpy matplotlib seaborn jupyter

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np
# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
# Settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
%matplotlib inline
print('Libraries imported successfully!')

In [ ]:

# Load data
df = pd.read_csv('/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv')
# First look
print('Dataset shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

## 1. Dataset Overview

In [ ]:
# 1.1 Column names and types
print('Column Information:')
df.info()

In [ ]:
# 1.2 Statistical summary
df.describe()

In [ ]:
# 1.3 Check for missing values
print('Missing Values:')
missing = df.isnull().sum()
print(missing[missing > 0])
# Fix TotalCharges if needed
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [ ]:
# 1.4 Target variable distribution
print('Churn Distribution:')
print(df['Churn'].value_counts())
print('\nChurn Percentage:')
print(df['Churn'].value_counts(normalize=True) * 100)

### Key Findings:
- Dataset has X customers
- Churn rate is Y%
- Z missing values in TotalCharges column
- Features include demographics, services, and account info

## 2. Numerical Features Analysis

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist(df['tenure'], bins=30, color='skyblue', edgecolor='black')
plt.xlabel('Tenure (months)')
plt.ylabel('Number of Customers')
plt.title('Distribution of Customer Tenure')
plt.subplot(1, 2, 2)
sns.boxplot(x='Churn', y='tenure', data=df)
plt.title('Tenure by Churn Status')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist(df['MonthlyCharges'], bins=30, color='lightcoral', edgecolor='black')
plt.xlabel('Monthly Charges ($)')
plt.ylabel('Number of Customers')
plt.title('Distribution of Monthly Charges')
plt.subplot(1, 2, 2)
sns.boxplot(x='Churn', y='MonthlyCharges', data=df)
plt.title('Monthly Charges by Churn Status')
plt.tight_layout()
plt.show()

In [ ]:
# Remove missing values for this plot
df_clean = df[df['TotalCharges'].notna()]
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.violinplot(x='Churn', y='TotalCharges', data=df_clean)
plt.title('Total Charges by Churn Status')
plt.subplot(1, 2, 2)
sns.scatterplot(x='tenure', y='TotalCharges', hue='Churn', data=df_clean,
alpha=0.5)
plt.title('Tenure vs Total Charges')
plt.tight_layout()
plt.show()

### Numerical Features Insights:
- Tenure: Customers with shorter tenure more likely to churn
- Monthly Charges: Higher charges correlate with higher churn
- Total Charges: Related to tenure (longer tenure = higher total charges)

## 3. Categorical Features Analysis

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
df['Contract'].value_counts().plot(kind='bar', color='steelblue')
plt.title('Contract Type Distribution')
plt.xlabel('Contract Type')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.subplot(1, 2, 2)
contract_churn = pd.crosstab(df['Contract'], df['Churn'], normalize='index') * 100
contract_churn.plot(kind='bar', stacked=False)
plt.title('Churn Rate by Contract Type')
plt.xlabel('Contract Type')
plt.ylabel('Percentage')
plt.legend(title='Churn')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.countplot(x='InternetService', hue='Churn', data=df)
plt.title('Churn by Internet Service Type')
plt.xlabel('Internet Service')
plt.xticks(rotation=45)
plt.subplot(1, 2, 2)
internet_churn = pd.crosstab(df['InternetService'], df['Churn'], normalize='index')
* 100
internet_churn['Yes'].plot(kind='bar', color='coral')
plt.title('Churn Rate by Internet Service')
plt.ylabel('Churn Rate (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
payment_churn = pd.crosstab(df['PaymentMethod'], df['Churn'], normalize='index') * 100
plt.figure(figsize=(10, 6))
payment_churn['Yes'].sort_values().plot(kind='barh', color='teal')
plt.title('Churn Rate by Payment Method')
plt.xlabel('Churn Rate (%)')
plt.ylabel('Payment Method')
plt.tight_layout()
plt.show()

## 4. Feature Correlations

In [ ]:
# Prepare numeric data
df_numeric = df.copy()
# Convert Yes/No to 1/0
binary_cols = ['Churn', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    if col in df_numeric.columns:
        df_numeric[col] = df_numeric[col].map({'Yes': 1, 'No': 0})
# Select only numeric columns
numeric_features = df_numeric.select_dtypes(include=[np.number])
# Correlation with Churn
plt.figure(figsize=(10, 8))
churn_corr = numeric_features.corr()['Churn'].sort_values(ascending=False)
churn_corr.plot(kind='barh', color='purple')
plt.title('Feature Correlation with Churn')
plt.xlabel('Correlation Coefficient')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 10))
correlation_matrix = numeric_features.corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 5. Key Insights and Findings
### High-Risk Customer Characteristics:
1. **Contract Type**: Month-to-month contracts have XX% churn rate (vs Y% for
yearly)
2. **Tenure**: Customers with <6 months tenure are high risk
3. **Charges**: Higher monthly charges correlate with higher churn
4. **Services**: Fiber optic internet users churn more than DSL
5. **Payment**: Electronic check users have higher churn
### Patterns Observed:
- [Add 3-5 specific observations from your analysis]
### Recommendations for Feature Engineering:
- Create 'TenureGroup' (0-12, 13-24, 25-48, 49+ months)
- Flag 'HighMonthlyCharges' (>$70)
- Count total services subscribed
- Create 'ChargesPerService' ratio
### Questions for Next Week:
- Which features should we include in the model?
- How to handle categorical variables?
- What about class imbalance (if churn rate is low)?

Report

### **What is this project about?**

This project looks at data from a telecommunications company to understand **customer churn**. "Churn" is just a business word for when customers stop using a service. By looking at patterns in the data, the goal is to predict which customers are likely to leave so the company can try to keep them.

### **The Data at a Glance**

* 
**Size**: There are **7,043 customers** in this list, and each customer has **21 different pieces of information** recorded about them.


* 
**Information types**: The data includes personal details (like age and gender), the services they use (like internet or phone), and their bills (how much they pay and how they pay it).



### **Important Discoveries **

The analysis found that certain groups of customers are much more likely to leave the company:

* 
**Monthly Contracts**: People who pay month-to-month are more likely to leave than those who sign a contract for a full year.


* 
**New Customers**: Customers who have been with the company for a very short time (less than 6 months) are in the "high risk" group for leaving.


* 
**Internet Type**: Interestingly, people using **Fiber optic** internet leave more often than those using DSL.


* 
**High Bills**: Customers with higher monthly bills are more likely to stop their service.


* 
**Payment Method**: People who pay with an **electronic check** tend to leave more than those using other methods like automatic bank transfers.



### **The Tools Used**

To find these patterns, the project used **Python**, a popular computer programming language for data science. Specifically, it used:

* 
**Pandas and NumPy**: To organize and clean the information.


* 
**Matplotlib and Seaborn**: To create charts and graphs that make the patterns easier to see.